# ĐỘT PHÁ #1 — Train recognizer Nôm trên Kaggle (student vượt thầy ở OOV/lỗi)

Train ResNet34 → softmax 1591-lớp trên nhãn tự-sinh; mỗi epoch in **student vs teacher** (+ OOV student, teacher-wrong recovery) trên test tách-sách. Đẩy `recognizer.best.pt` lên HF.

**Trước khi Run:** (1) Upload `kaggle_rec_pkg/` làm Dataset; (2) Add Input → chọn nó; (3) Settings → Accelerator = **GPU T4 x2** (KHÔNG P100), Internet = On; (4) (HF) Add-ons → Secrets → `HF_TOKEN`; (5) Run All.

In [ ]:
import os, glob, torch, csv, collections
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - bật Accelerator!')
hits = glob.glob('/kaggle/input/**/labels.csv', recursive=True)
assert hits, 'Không thấy labels.csv — đã Add đúng Dataset kaggle_rec_pkg chưa?'
ROOT = os.path.dirname(hits[0])
print('ROOT =', ROOT, '| files:', os.listdir(ROOT)[:8])
print('rows by tier:', dict(collections.Counter(r['tier'] for r in csv.DictReader(open(ROOT+'/labels.csv')))))

## A) Train STUDENT trên nhãn ĐỒNG THUẬN (~1–2h / T4). Lưu best theo student-acc; in OOV/recovery mỗi epoch.

In [ ]:
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN'); print('HF_TOKEN: loaded')
except Exception as e:
    print('HF_TOKEN: none ->', e)
HF_REPO = ''   # vd 'mdnt571/nom-recognizer' (để trống = chỉ Kaggle Output)
init = ROOT + '/encoder_best.pt'
cmd = [sys.executable, ROOT + '/train_recognizer.py', '--root', ROOT, '--target', 'consensus',
       '--arch', 'resnet34', '--img', '160', '--epochs', '30', '--batch', '128', '--workers', '2',
       '--out', '/kaggle/working/recognizer.pt']
if os.path.exists(init): cmd += ['--init', init]
if HF_REPO and os.environ.get('HF_TOKEN'): cmd += ['--hf-repo', HF_REPO]
subprocess.run(cmd, check=True)

## Đọc HEADLINE
OOV student > 0 (teacher = 0) → thắng tuyệt đối (chữ Nôm-thuần OCR không xuất được). teacher-wrong recovery = % lỗi thầy student sửa. Tổng: thầy thường ≥ ở chữ phổ biến (GOLD thiên lệch) → ĐỪNG tuyên thắng tổng.

In [ ]:
import torch
ck = torch.load('/kaggle/working/recognizer.best.pt', map_location='cpu')
print('classes:', len(ck['classes']), '| best TEST:', ck.get('test'))
print('Kéo về: huggingface-cli download <HF_REPO> recognizer.best.pt --local-dir evaluation/ver_new/nom_recognizer')

## B) (tùy chọn) Control nhãn OCR thô → bảng 3-chiều A/B/C (A đồng thuận · B OCR thô · C teacher).

In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, ROOT+'/train_recognizer.py','--root',ROOT,'--target','ocr',
#   '--arch','resnet34','--img','160','--epochs','30','--batch','128','--workers','2',
#   '--init',ROOT+'/encoder_best.pt','--out','/kaggle/working/recognizer_ocr.pt'], check=True)